<a href="https://colab.research.google.com/github/AshakUmesh/Learn-Python/blob/main/Ciphers/IS_Ciphers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import random
import math
from itertools import combinations

# ---------- Settings ----------
random.seed(7)   # fixed seed for reproducibility
TEXT_LEN  = 15   # plaintext length
TRIALS    = 8    # confusion trials

ALPHA = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
ALPHANUM = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
ADFGVX_SIGNS = "ADFGVX"

def clean_text(s, alphabet):
    return "".join(ch for ch in s.upper() if ch in alphabet)

# ---------- Hungarian (min-cost) ----------
def hungarian_min_cost(cost):
    n = len(cost)
    u = [0] * (n+1)
    v = [0] * (n+1)
    p = [0] * (n+1)
    way = [0] * (n+1)

    for i in range(1, n+1):
        p[0] = i
        j0 = 0
        minv = [float('inf')] * (n+1)
        used = [False] * (n+1)
        while True:
            used[j0] = True
            i0 = p[j0]
            delta = float('inf'); j1 = 0
            for j in range(1, n+1):
                if not used[j]:
                    cur = cost[i0-1][j-1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur
                        way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]; j1 = j
            for j in range(n+1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while True:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break

    assignment = [0] * (n+1)
    for j in range(1, n+1):
        assignment[p[j]] = j
    return assignment[1:]

def relabel_invariant_distance(a, b, alphabet):
    L = min(len(a), len(b))
    idx = {ch:i for i,ch in enumerate(alphabet)}
    n = len(alphabet)
    counts = [[0]*n for _ in range(n)]
    for i in range(L):
        counts[idx[a[i]]][idx[b[i]]] += 1
    maxcnt = max((c for row in counts for c in row), default=0)
    cost = [[maxcnt - counts[i][j] for j in range(n)] for i in range(n)]
    assign = hungarian_min_cost(cost)
    best_matches = sum(counts[i][assign[i]-1] for i in range(n))
    return 1.0 - (best_matches / L if L else 0.0)

# ---------- Ciphers ----------
def shift_encrypt(pt, k):
    return "".join(ALPHA[(ord(ch)-65 + k) % 26] for ch in pt)

def random_substitution_key():
    perm = list(ALPHA); random.shuffle(perm)
    return dict(zip(ALPHA, perm))

def substitution_encrypt(pt, keymap):
    return "".join(keymap[ch] for ch in pt)

def vigenere_encrypt(pt, key):
    key = clean_text(key, ALPHA) or "A"
    return "".join(ALPHA[(ord(ch)-65 + (ord(key[i%len(key)])-65)) % 26] for i,ch in enumerate(pt))

def rand_invertible_hill2_key():
    while True:
        a,b,c,d = [random.randrange(26) for _ in range(4)]
        det = (a*d - b*c) % 26
        if math.gcd(det, 26) == 1:
            return (a,b,c,d)

def hill2_encrypt(pt, mat):
    if len(pt) % 2 == 1:
        pt += "X"
    a,b,c,d = mat
    out = []
    for i in range(0, len(pt), 2):
        x = ord(pt[i]) - 65; y = ord(pt[i+1]) - 65
        out.append(ALPHA[(a*x + b*y) % 26])
        out.append(ALPHA[(c*x + d*y) % 26])
    return "".join(out)

def columnar_order(key):
    enumerated = list(enumerate(key))
    return [i for i,_ in sorted(enumerated, key=lambda t: (t[1], t[0]))]

def columnar_transpose(text, key):
    if not key:
        return text
    n = len(key)
    pad_len = (n - (len(text) % n)) % n
    text += "X" * pad_len
    rows = [text[i:i+n] for i in range(0, len(text), n)]
    col_order = columnar_order(key)
    out = []
    for col in col_order:
        for r in rows:
            out.append(r[col])
    return "".join(out)

def adfgvx_polybius_key():
    seq = list(ALPHANUM); random.shuffle(seq)
    table = {}
    for idx, ch in enumerate(seq):
        r = idx // 6; c = idx % 6
        table[ch] = ADFGVX_SIGNS[r] + ADFGVX_SIGNS[c]
    return table

def adfgvx_encrypt(pt, sub_table, trans_key):
    frac = "".join(sub_table[ch] for ch in pt)
    return columnar_transpose(frac, trans_key)

# ---------- Metrics ----------
def spread_score(changed_positions, total_len):
    if len(changed_positions) <= 1:
        return 0.0
    dists = []
    for i, j in combinations(changed_positions, 2):
        dists.append(abs(j - i))
    avg_d = sum(dists) / len(dists)
    max_avg_d = total_len / 3
    return min(1.0, avg_d / max_avg_d) if max_avg_d > 0 else 0.0

def diffusion_score(cipher_fn, key, plaintext, perturb_char='Z'):
    C = cipher_fn(plaintext, key)
    L = len(C)
    if L == 0:
        return 0.0
    change_fracs = []
    spreads = []
    for pos in range(len(plaintext)):
        orig = plaintext[pos]
        new_ch = perturb_char if perturb_char != orig else 'Y'
        P2 = plaintext[:pos] + new_ch + plaintext[pos+1:]
        C2 = cipher_fn(P2, key)
        changed = [i for i in range(min(len(C), len(C2))) if C[i] != C2[i]]
        cf = len(changed) / L
        sp = spread_score(changed, L)
        change_fracs.append(cf)
        spreads.append(sp)
    return 0.7 * (sum(change_fracs)/len(change_fracs)) + 0.3 * (sum(spreads)/len(spreads))

def tweak_key(cipher_name, key):
    if cipher_name == "shift":
        return (key + 1) % 26
    if cipher_name == "substitution":
        k = key.copy()
        a, b = random.sample(ALPHA, 2)
        k[a], k[b] = k[b], k[a]
        return k
    if cipher_name == "vigenere":
        idx = random.randrange(len(key))
        delta = random.randrange(1, 26)
        ch = ALPHA[(ord(key[idx]) - 65 + delta) % 26]
        return key[:idx] + ch + key[idx+1:]
    if cipher_name == "hill2":
        a,b,c,d = key
        which = random.choice([0,1,2,3])
        vec = [a,b,c,d]
        vec[which] = (vec[which] + 1) % 26
        for _ in range(100):
            aa,bb,cc,dd = vec
            det = (aa*dd - bb*cc) % 26
            if math.gcd(det, 26) == 1:
                return (aa,bb,cc,dd)
            vec[which] = (vec[which] + 1) % 26
        return rand_invertible_hill2_key()
    if cipher_name == "transposition":
        if len(key) < 2:
            return key
        pos = random.randrange(len(key)-1)
        k_list = list(key)
        k_list[pos], k_list[pos+1] = k_list[pos+1], k_list[pos]
        return "".join(k_list)
    if cipher_name == "adfgvx":
        if random.random() < 0.5:
            sub_table, trans_key = key
            a, b = random.sample(ALPHANUM, 2)
            new_map = dict(sub_table)
            new_map[a], new_map[b] = new_map[b], new_map[a]
            return (new_map, trans_key)
        else:
            sub_table, trans_key = key
            if len(trans_key) < 2:
                return (sub_table, trans_key)
            pos = random.randrange(len(trans_key)-1)
            k_list = list(trans_key)
            k_list[pos], k_list[pos+1] = k_list[pos+1], k_list[pos]
            return (sub_table, "".join(k_list))
    raise ValueError("Unknown cipher")

def confusion_score(cipher_name, cipher_fn, key, plaintext, trials=TRIALS):
    base = cipher_fn(plaintext, key)
    if len(base) == 0:
        return 0.0
    out_alpha = ADFGVX_SIGNS if cipher_name == "adfgvx" else ALPHA
    vals = []
    for _ in range(trials):
        tweaked = tweak_key(cipher_name, key)
        C2 = cipher_fn(plaintext, tweaked)
        vals.append(relabel_invariant_distance(base, C2, out_alpha))
    return sum(vals) / len(vals)

# ---------- Wrappers ----------
def cipher_wrapper(name):
    if name == "shift":
        k = random.randrange(1, 26)
        return lambda pt, K=k: shift_encrypt(pt, K), k
    if name == "substitution":
        k = random_substitution_key()
        return lambda pt, K=k: substitution_encrypt(pt, K), k
    if name == "vigenere":
        keylen = 6
        key = "".join(random.choice(ALPHA) for _ in range(keylen))
        return lambda pt, K=key: vigenere_encrypt(pt, K), key
    if name == "hill2":
        k = rand_invertible_hill2_key()
        return lambda pt, K=k: hill2_encrypt(pt, K), k
    if name == "transposition":
        keylen = 7
        key = "".join(random.choice(ALPHA) for _ in range(keylen))
        return lambda pt, K=key: columnar_transpose(pt, K), key
    if name == "adfgvx":
        sub = adfgvx_polybius_key()
        keylen = 7
        tkey = "".join(random.choice(ADFGVX_SIGNS) for _ in range(keylen))
        return lambda pt, K=(sub, tkey): adfgvx_encrypt(pt, K[0], K[1]), (sub, tkey)
    raise ValueError("Unknown cipher name")

# ---------- Single Plaintext Experiment ----------
def run_experiment_single(text_len=TEXT_LEN, trials=TRIALS):
    plaintext = "".join(random.choice(ALPHA) for _ in range(text_len))
    print("Plaintext (first 60 chars):", plaintext[:60] + "...\n")
    results = {}
    names = ["shift", "substitution", "vigenere", "hill2", "transposition", "adfgvx"]
    for name in names:
        enc_fn, key = cipher_wrapper(name)
        d = diffusion_score(enc_fn, key, plaintext)
        c = confusion_score(name, enc_fn, key, plaintext, trials=trials)
        results[name] = {"diffusion": d, "confusion": c}
    return results

def main():
    results = run_experiment_single()
    print("Results on SINGLE plaintext:")
    for name, vals in results.items():
        print(f"{name:15s} | diffusion: {vals['diffusion']:.4f} | confusion: {vals['confusion']:.4f}")

if __name__ == "__main__":
    main()


Plaintext (first 60 chars): KEMUBCRDLSBQGBC...

Results on SINGLE plaintext:
shift           | diffusion: 0.0467 | confusion: 0.0000
substitution    | diffusion: 0.0467 | confusion: 0.0000
vigenere        | diffusion: 0.0467 | confusion: 0.1083
hill2           | diffusion: 0.1171 | confusion: 0.2109
transposition   | diffusion: 0.0333 | confusion: 0.1131
adfgvx          | diffusion: 0.1914 | confusion: 0.1214
